# ViDoRe V3 Industrial · 独立案例
只使用官方 OCR 文本与页面来源，不是端到端视觉理解；页级相关性不能自动转成多页充分性。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', '')  # Set your own repository URL to enable Git sync.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'pilot'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    subprocess.run([sys.executable, '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)], check=True)


In [ ]:
# Credentials are read from Colab Secrets or environment; never printed or saved.
if 'google.colab' in sys.modules:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY'):
        if not os.environ.get(key):
            try:
                value = userdata.get(key)
                if value: os.environ[key] = value
            except Exception:
                pass


In [ ]:
from sa_jev.industrial import download_ocr
pages, queries, qrels = download_ocr(ROOT/'industrial')

In [ ]:
from sa_jev.industrial import page_pool, evaluate_industrial
from sa_jev.retrieval import make_lightrag, ingest_corpus, retrieve_once, PoolCache
from sa_jev.schema import Evidence
from sa_jev.experiments import load_method
from sa_jev.reporting import export_report
from sa_jev.io import write_jsonl
rag = await make_lightrag(ROOT/'industrial/index')
try:
    docs = [Evidence(p['id'],p['title'],p['text'],p['source']) for p in pages if p['text'].strip()]
    binding = await ingest_corpus(rag, docs, ROOT/'industrial/index/manifest.json', config['retrieval'])
    selected_queries = queries[:8] if MODE=='pilot' else queries
    cache = PoolCache(ROOT/'industrial/chunk_pools')
    pools = {}
    for q in selected_queries:
        qid = 'vidore:'+str(q['query_id'])
        chunks = await retrieve_once(rag, qid, q['query'], 64, cache, corpus_fingerprint=binding, source_registry={e.id:e for e in docs})
        pools[qid] = page_pool(qid, chunks, pages, 32)
finally:
    await rag.finalize_storages()
model, _ = load_method(ROOT, config, 'full', 17)
rows = evaluate_industrial(model, selected_queries, qrels, pools)
write_jsonl(ROOT/'results/industrial/raw.jsonl', rows)
export_report(rows, ROOT/'results/industrial', ('page_ndcg5','page_recall5','seconds'))

In [ ]:
# Export sourced successes AND failures; never fabricate a case if one category is absent.
for success in (True, False):
    subset = [r for r in rows if r['retrieval_success'] is success]
    print('success' if success else 'failure', subset[:2])